# 高维数据的处理

> Shinging -> Min-Hashing -> Locality-Sensitive-Hashing

- 图片数据
- 包含相同文字的页:冗余检测,主题分类
- 购买相同产品的顾客
- 推荐和搜索

图片数据的大数据处理模式:
- 使用神经网络对图像进行特征化嵌入,大概4000个维度
- 给定一个query模式,快速找到近似图片

> 淘宝,拍照找图


高维数据的处理范式:
- 给定高维数据点$x_1,x_2$
- 给定距离计算方式$d(x_1,x_2)$
- 找到所有$d(x_i,x_j)\le s$
- 在$O(N)$范围内实现

## LSH 

- LSH代表一类技术
- 通常来说会将元素通过hash丢进不同的buckets
- upside :正确的设计,应该只需要检查一部数据
- downside : 负采样

## 查找相似的文档:

假设:在 N = 1m 的文章中找到近似的文档
- 通过情况下我们需要计算每一对之间的相似度
- 这是一个`N(N-1)/2`的计算

计算文档相似性的关键步骤:
- Shingling 将文档转换为一个二进制的表示
- Min-Hashing 在保证相似性的情况下,将大型数据集转换为一次签名
- Locality-Sensitive Hashing 重点关注可能相似的文章之间的相似性计算

idea就是将文档hashing到一个bucket中,然后对一个hucket中的文档进行相似度计算

###  Shingling

将一个文档转换成一个集合.集合中包含k个tokens,tokens可以是characters,words或者其他取决于应用

为了压缩长Shingles,可以将他们压缩到4Bytes,可以使用一组hahs值来代表文档的 k-shingles 集合.

Shingling的好处:
- 通常来说有相同shingle的文档是相似的
- 改变一个词只会影响 k-shingles中的 k-1距离内的词

> 这个算法好像挺适合中文 k = 2  
> D = 你好2020!  
> S(D) = {你好,好2,20,02,20}  
> Hash,h(D) = {1,5,7,9,7}  

### 相似度量 Jaccard

- Jaccard相似度量
$sim(D_1,D_2) = \frac{C_1\cup C_2}{C_1\cap C_2}$
- Jaccard距离
$d(D_1,D_2) =1 - \frac{C_1\cup C_2}{C_1\cap C_2}$

### 集合转换矩阵

$mat = 文档\times elements(shingles)$

其中$mat[e][s]=1$代表e在文档中,*这是不是有点类似词袋模型*

### Min Hashing

签名生成(特征压缩)

key idea: hash 每一行到一个小的 signature h(C) ,此时$sim(C_1,C_2)$近似$sim'(h(C_1),h(C_2))$ 

此时目标就是找到这个 hash 函数
- sim(C_1,C_2) 高, $h(C_1)=h(c_2)$
- sim(C_1,C_2) 低, $h(C_1)\neq h(c_2)$
 
假设有这样的hash函数:那么相似的文章就会被丢到相同的bucket

这样的`Hash函数`依赖于`相似度量`方式,不一定所有的相似度量方式都有合适的hash函数
$$
h_\pi (C) = min_\pi \pi(C)
$$

表示每一行第一个1的位置的词的hash码,这样每一个文档的多维表示就被压缩成了1维

证明略:$Pr[h_\pi (C_1)=h_\pi (C_2)]=sim(C_1,C_2)$

使用多随机 `Shingling-hash` 计算相似度

随机数量越多$Pr[h_\pi (C_1)=h_\pi (C_2)]=sim(C_1,C_2)$越成立


### Locality Sensitive Hashing

一个 bucket 内的相似度计算

目标: 在Jaccord相似度小于0.8的相似文档

算法: LSH

方法: 将相似的列hash到一个bucket,然后在bucket中组件候对

将 M : hashvalue\*documents 中的 hashvalue 分带(b),没条连续带中有 r 行

对每一个带进行一次bucket划分,相桶的可以计算相似度

主要也就是 band 相同

> 相似度 > 0.8 的文档对,有99.965%的概率通过该算法发现到  
> 相似度 < 0.3 的文档对,有4.74%的概率通过该算法被计算到



### LSH算法的调优空间
- M 的数量 Min-hashes
- b 的大小
- 没个 b 内的 r 数量, 其中 M = b*r
- 阈值 s 

```
s   1-(1-s^r)^b
0.2  0.006
0.3  0.047
0.4  0.186
0.5  0.470
0.6  0.802
0.7  0.975
0.8  0.9996
```
